# Model Training & Selection - Workout Monitoring System

## Purpose

**WHY train multiple models?**

1. **No single best algorithm**: Different models excel at different aspects
2. **Comparison baseline**: Need reference points to evaluate deep learning
3. **Understand trade-offs**: Speed vs accuracy, interpretability vs performance
4. **Ensemble potential**: Combining models can improve robustness

### Models We'll Train:

| Model | WHY This Model |
|-------|---------------|
| **Random Forest** | Baseline, feature importance, handles non-linear relationships |
| **SVM (RBF)** | Strong for high-dimensional data, robust margins |
| **LSTM** | Captures temporal dependencies in exercise sequences |

---

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                           f1_score, classification_report, confusion_matrix)
from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✓ Libraries imported")

## 1. Load Balanced Data

**WHY use balanced data?**
- We applied SMOTE in the previous notebook to handle class imbalance
- Training on balanced data prevents model bias toward majority class
- Test set remains imbalanced (realistic distribution)

In [ ]:
# Load balanced training data and test data
train_data = np.load('../data/balanced/train_balanced.npz')
test_data = np.load('../data/balanced/test.npz')

X_train = train_data['X']
y_train = train_data['y']
X_test = test_data['X']
y_test = test_data['y']

print(f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTraining class distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for c, count in zip(unique, counts):
    print(f"  Class {c}: {count} samples ({count/len(y_train)*100:.1f}%)")

In [ ]:
# Normalize features
# WHY: Many ML algorithms are sensitive to feature scales
# StandardScaler: zero mean, unit variance

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # Use SAME scaler fitted on training

print("✓ Features normalized with StandardScaler")
print(f"  Mean before: {X_train.mean():.3f}, Std: {X_train.std():.3f}")
print(f"  Mean after: {X_train_scaled.mean():.6f}, Std: {X_train_scaled.std():.3f}")

## 2. Model 1: Random Forest

### WHY Random Forest?

1. **Ensemble of decision trees**: Reduces overfitting through averaging
2. **Feature importance**: Tells us WHICH features matter most
3. **No scaling required**: Tree-based methods are scale-invariant
4. **Handles non-linear relationships**: Captures complex patterns
5. **Robust baseline**: Well-understood, reliable performance

### How It Works:
- Trains multiple decision trees on random subsets of data
- Each tree votes on the class
- Final prediction = majority vote

### Hyperparameters:
- `n_estimators`: Number of trees (more = better but slower)
- `max_depth`: Maximum tree depth (prevent overfitting)
- `min_samples_split`: Minimum samples to split a node

In [ ]:
# Train Random Forest
print("Training Random Forest...")
print("-" * 40)

rf = RandomForestClassifier(
    n_estimators=100,     # 100 trees
    max_depth=15,         # Prevent overfitting
    min_samples_split=5,  # Minimum samples to split
    random_state=42,
    n_jobs=-1             # Use all CPU cores
)

# Note: RF doesn't need scaled features, but we use them for consistency
rf.fit(X_train, y_train)

# Evaluate
train_acc = rf.score(X_train, y_train)
test_acc = rf.score(X_test, y_test)

print(f"✓ Training complete")
print(f"  Training accuracy: {train_acc:.3f}")
print(f"  Test accuracy: {test_acc:.3f}")

# Check for overfitting
if train_acc - test_acc > 0.1:
    print(f"\n⚠️  Warning: Possible overfitting (gap: {train_acc - test_acc:.3f})")
else:
    print(f"\n✓ Good generalization (gap: {train_acc - test_acc:.3f})")

In [ ]:
# Feature Importance Analysis
# WHY: Understand which features the model relies on most

importances = rf.feature_importances_
indices = np.argsort(importances)[::-1][:20]  # Top 20

plt.figure(figsize=(12, 6))
plt.bar(range(20), importances[indices], color='steelblue', alpha=0.8)
plt.xlabel('Feature Index')
plt.ylabel('Importance Score')
plt.title('Top 20 Most Important Features (Random Forest)', fontsize=14, fontweight='bold')
plt.xticks(range(20), indices)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 Top 10 Most Important Features:")
for i, idx in enumerate(indices[:10], 1):
    print(f"  {i}. Feature {idx}: {importances[idx]:.4f}")

## 3. Model 2: Support Vector Machine (SVM)

### WHY SVM with RBF Kernel?

1. **High-dimensional effectiveness**: Works well with 72 features
2. **Maximum margin classifier**: Finds optimal decision boundary
3. **Kernel trick**: RBF captures non-linear relationships
4. **Robust to outliers**: Margin-based, ignores far-away points
5. **Theoretical foundation**: Strong mathematical guarantees

### How It Works:
- Finds hyperplane that maximizes margin between classes
- RBF kernel maps data to higher dimensions
- Only support vectors (boundary points) matter

### Key Hyperparameters:
- `C`: Regularization (higher = more complex boundary)
- `gamma`: RBF kernel width (higher = more local influence)

In [ ]:
# Train SVM with RBF kernel
print("Training SVM with RBF kernel...")
print("-" * 40)

svm = SVC(
    kernel='rbf',
    C=10,              # Regularization parameter
    gamma='scale',     # Auto-scale based on features
    random_state=42,
    probability=True   # Enable probability estimates
)

# SVM NEEDS scaled features
svm.fit(X_train_scaled, y_train)

# Evaluate
train_acc = svm.score(X_train_scaled, y_train)
test_acc = svm.score(X_test_scaled, y_test)

print(f"✓ Training complete")
print(f"  Training accuracy: {train_acc:.3f}")
print(f"  Test accuracy: {test_acc:.3f}")
print(f"  Number of support vectors: {sum(svm.n_support_)}")

## 4. Model Comparison

**WHY compare multiple metrics?**

| Metric | What It Measures | When It Matters |
|--------|-----------------|----------------|
| **Accuracy** | Overall correctness | Balanced datasets |
| **Precision** | "Of predicted positives, how many correct?" | Cost of false positives high |
| **Recall** | "Of actual positives, how many found?" | Missing positives costly |
| **F1-Score** | Harmonic mean of precision/recall | Balance both |

For workout monitoring:
- **High precision**: Don't falsely tell someone their form is correct
- **High recall**: Catch all incorrect forms to prevent injury

In [ ]:
# Evaluate all models
models = {
    'Random Forest': (rf, X_test),
    'SVM (RBF)': (svm, X_test_scaled)
}

results = []

for name, (model, X) in models.items():
    y_pred = model.predict(X)
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'Recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, average='weighted', zero_division=0)
    })

results_df = pd.DataFrame(results)
print("Model Comparison Results:")
print("=" * 70)
print(results_df.to_string(index=False))

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of metrics
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics))
width = 0.35

for i, row in results_df.iterrows():
    values = [row[m] for m in metrics]
    offset = (i - 0.5) * width
    axes[0].bar(x + offset, values, width, label=row['Model'], alpha=0.8)

axes[0].set_xlabel('Metric')
axes[0].set_ylabel('Score')
axes[0].set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].legend()
axes[0].set_ylim(0, 1.1)
axes[0].grid(axis='y', alpha=0.3)

# Confusion Matrix for best model
best_model_name = results_df.loc[results_df['F1-Score'].idxmax(), 'Model']
best_model, X_best = models[best_model_name]
y_pred_best = best_model.predict(X_best)

cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title(f'Confusion Matrix: {best_model_name}', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

print(f"\n🏆 Best Model: {best_model_name}")
print(f"   F1-Score: {results_df.loc[results_df['F1-Score'].idxmax(), 'F1-Score']:.3f}")

## 5. Cross-Validation

**WHY use cross-validation?**

1. **More reliable estimate**: Single train/test split may be lucky/unlucky
2. **Use all data**: Every sample used for both training and validation
3. **Variance estimate**: See how much performance varies across folds
4. **Detect overfitting**: High variance suggests overfitting

### How 5-Fold CV Works:
- Split data into 5 equal parts
- Train on 4 parts, validate on 1 part
- Repeat 5 times (each part is validation once)
- Average the 5 scores

In [ ]:
# Cross-validation
print("Running 5-fold Cross-Validation...")
print("-" * 40)

cv_results = []

for name, (model, _) in models.items():
    # Use scaled data for all (RF is scale-invariant, SVM needs it)
    X_all = np.vstack([X_train_scaled, X_test_scaled])
    y_all = np.hstack([y_train, y_test])
    
    scores = cross_val_score(model, X_all, y_all, cv=5, scoring='f1_weighted')
    
    cv_results.append({
        'Model': name,
        'Mean F1': scores.mean(),
        'Std F1': scores.std(),
        'Min F1': scores.min(),
        'Max F1': scores.max()
    })
    
    print(f"\n{name}:")
    print(f"  Scores: {scores}")
    print(f"  Mean: {scores.mean():.3f} (+/- {scores.std()*2:.3f})")

cv_df = pd.DataFrame(cv_results)
print("\n" + "=" * 50)
print("Cross-Validation Summary:")
print(cv_df.to_string(index=False))

## 6. Save Best Model

**WHY save the model?**
- Avoid retraining for deployment
- Reproducibility of results
- Version control for models

In [ ]:
import joblib
import os

# Create models directory
os.makedirs('../models', exist_ok=True)

# Save best model
best_idx = results_df['F1-Score'].idxmax()
best_model_name = results_df.loc[best_idx, 'Model']
best_model = models[best_model_name][0]

# Save model and scaler
joblib.dump(best_model, '../models/best_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

print(f"✓ Saved best model: {best_model_name}")
print(f"  Location: models/best_model.pkl")
print(f"  Scaler: models/scaler.pkl")

## 7. Summary & Key Takeaways

### Model Performance Summary

| Model | Strengths | Weaknesses | Best For |
|-------|-----------|------------|----------|
| **Random Forest** | Feature importance, no scaling needed, robust | May overfit, slower inference | Interpretability, baseline |
| **SVM (RBF)** | High-dim effective, max margin | Scaling required, no feature importance | High-dimensional classification |
| **LSTM** (not trained here) | Temporal patterns, variable length | Needs more data, slower training | Sequence classification |

### Why Our Choice Matters

For workout monitoring:
- **Safety is paramount**: High precision prevents false "good form" signals
- **User experience**: Low latency needed for real-time feedback
- **Interpretability**: Understanding why a form is incorrect helps users improve

### Next Steps
1. **Detailed evaluation** in notebook 06
2. **Error analysis** to understand failure cases
3. **Deployment** for real-time inference

---

**Next Notebook**: `06_model_evaluation.ipynb` - Comprehensive performance analysis

In [ ]:
print("\n" + "=" * 70)
print("MODEL TRAINING COMPLETE!")
print("=" * 70)
print("\n✓ Trained Random Forest and SVM models")
print("✓ Compared performance metrics")
print("✓ Performed cross-validation")
print("✓ Saved best model for deployment")
print("\n📊 Move to next notebook: 06_model_evaluation.ipynb")